# Small Poetry Next-Word Model: Full Training Pipeline

This notebook trains a **small, simple baseline** model that predicts the next word from context.

It includes:
- data download from the web
- preprocessing and tokenization
- building train/validation samples
- training a lightweight GRU next-word model
- evaluation with validation perplexity
- saving artifacts to `model/artifacts`
- inference with optional rhyme and tempo constraints


## Data Source (web)

Primary dataset used in this notebook:
- Gutenberg Poetry Corpus (Allison Parrish):
  - Repo: https://github.com/aparrish/gutenberg-poetry-corpus
  - Direct corpus file (NDJSON.GZ): https://static.decontextualize.com/gutenberg-poetry-v001.ndjson.gz

The corpus is described as poetry lines extracted from Project Gutenberg (public-domain texts in the US).


In [1]:
# If needed, install dependencies (run once):
%pip install -q torch tqdm


In [2]:
import gzip
import json
import math
import random
import re
import urllib.request
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


In [3]:
# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Resolve project paths whether notebook is run from repo root or from model/
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'model' else cwd
MODEL_DIR = PROJECT_ROOT / 'model'
DATA_DIR = MODEL_DIR / 'data'
ARTIFACT_DIR = MODEL_DIR / 'artifacts'

DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'DEVICE       = {DEVICE}')


PROJECT_ROOT = /content
DEVICE       = cpu


In [4]:
CORPUS_URL = 'https://static.decontextualize.com/gutenberg-poetry-v001.ndjson.gz'
CORPUS_PATH = DATA_DIR / 'gutenberg-poetry-v001.ndjson.gz'


def download_file(url: str, destination: Path) -> None:
    if destination.exists():
        print(f'Found existing file: {destination}')
        return

    print(f'Downloading {url} -> {destination}')
    urllib.request.urlretrieve(url, destination)
    print('Download complete.')


download_file(CORPUS_URL, CORPUS_PATH)


Download complete.


In [5]:
# Parsing and lightweight cleaning
WORD_RE = re.compile(r"[a-z']+")


def normalize_line(text: str) -> str:
    text = text.lower()
    text = text.replace('’', "'").replace('`', "'")
    text = re.sub(r"[^a-z'\s-]", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def iter_token_lines(path: Path, max_rows: int | None = 200_000):
    with gzip.open(path, mode='rt', encoding='utf-8') as handle:
        for row_idx, row in enumerate(handle):
            if max_rows is not None and row_idx >= max_rows:
                break

            obj = json.loads(row)
            line = normalize_line(obj.get('s', ''))
            tokens = WORD_RE.findall(line)

            # keep moderately short poetic lines
            if 4 <= len(tokens) <= 16:
                yield tokens


MAX_ROWS = 220_000  # subset for a faster first model
lines = list(tqdm(iter_token_lines(CORPUS_PATH, max_rows=MAX_ROWS), total=MAX_ROWS))
print(f'Collected {len(lines):,} usable lines')


  0%|          | 0/220000 [00:00<?, ?it/s]

Collected 216,657 usable lines


In [6]:
# Build vocabulary and (context -> next-word) training samples
CONTEXT_SIZE = 6
MIN_FREQ = 3
MAX_VOCAB = 20_000
MAX_SAMPLES = 350_000

counter = Counter(token for line in lines for token in line)
most_common = [w for w, c in counter.most_common(MAX_VOCAB) if c >= MIN_FREQ]

SPECIAL_TOKENS = ['<pad>', '<unk>']
itos = SPECIAL_TOKENS + most_common
stoi = {token: idx for idx, token in enumerate(itos)}

PAD_ID = stoi['<pad>']
UNK_ID = stoi['<unk>']


def encode_word(word: str) -> int:
    return stoi.get(word, UNK_ID)


samples: list[tuple[list[int], int]] = []
for line in lines:
    ids = [encode_word(t) for t in line]
    for i in range(1, len(ids)):
        context = ids[max(0, i - CONTEXT_SIZE):i]
        if len(context) < CONTEXT_SIZE:
            context = [PAD_ID] * (CONTEXT_SIZE - len(context)) + context
        target = ids[i]
        samples.append((context, target))

        if len(samples) >= MAX_SAMPLES:
            break
    if len(samples) >= MAX_SAMPLES:
        break

print(f'Vocabulary size: {len(itos):,}')
print(f'Training samples: {len(samples):,}')


Vocabulary size: 20,002
Training samples: 350,000


In [7]:
# Train/validation split
random.shuffle(samples)
split_idx = int(0.9 * len(samples))
train_samples = samples[:split_idx]
val_samples = samples[split_idx:]

print(f'Train samples: {len(train_samples):,}')
print(f'Val samples:   {len(val_samples):,}')


class NextWordDataset(Dataset):
    def __init__(self, pairs: list[tuple[list[int], int]]):
        self.pairs = pairs

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int):
        context, target = self.pairs[idx]
        return torch.tensor(context, dtype=torch.long), torch.tensor(target, dtype=torch.long)


train_ds = NextWordDataset(train_samples)
val_ds = NextWordDataset(val_samples)

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)


Train samples: 315,000
Val samples:   35,000


In [8]:
# Small baseline model (simple but trainable)
class SmallPoetryGRU(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int = 96, hidden_dim: int = 160):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x)
        _, hidden = self.gru(emb)
        logits = self.output(hidden[-1])
        return logits


model = SmallPoetryGRU(vocab_size=len(itos)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

print(model)


SmallPoetryGRU(
  (embedding): Embedding(20002, 96, padding_idx=0)
  (gru): GRU(96, 160, batch_first=True)
  (output): Linear(in_features=160, out_features=20002, bias=True)
)


In [9]:
# Training loop
EPOCHS = 4


def evaluate(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    total_loss = 0.0
    total_count = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * y.size(0)
            total_count += y.size(0)
    return total_loss / max(total_count, 1)


for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    running_count = 0

    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for x, y in progress:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        running_count += y.size(0)
        progress.set_postfix(train_loss=f'{running_loss / running_count:.4f}')

    train_loss = running_loss / max(running_count, 1)
    val_loss = evaluate(model, val_loader)
    val_ppl = math.exp(val_loss)
    print(f'Epoch {epoch}: train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_ppl={val_ppl:.2f}')


Epoch 1/4:   0%|          | 0/1231 [00:00<?, ?it/s]

Epoch 1: train_loss=6.7401 | val_loss=6.3555 | val_ppl=575.64


Epoch 2/4:   0%|          | 0/1231 [00:00<?, ?it/s]

Epoch 2: train_loss=5.9161 | val_loss=6.1586 | val_ppl=472.76


Epoch 3/4:   0%|          | 0/1231 [00:00<?, ?it/s]

Epoch 3: train_loss=5.3012 | val_loss=6.1133 | val_ppl=451.82


Epoch 4/4:   0%|          | 0/1231 [00:00<?, ?it/s]

Epoch 4: train_loss=4.7780 | val_loss=6.1428 | val_ppl=465.38


In [10]:
# Save model weights and vocabulary as separate files
model_path = ARTIFACT_DIR / 'small_poetry_gru.pt'
vocab_json_path = ARTIFACT_DIR / 'vocabulary.json'
vocab_tsv_path = ARTIFACT_DIR / 'vocabulary.tsv'
default_model_vocab_path = MODEL_DIR / 'vocabulary.json'

# Human-friendly vocabulary format
vocab_payload = {
    'format': 'poetry_vocabulary_v1',
    'size': len(itos),
    'special_tokens': SPECIAL_TOKENS,
    'tokens': itos,
}

vocab_json_path.write_text(
    json.dumps(vocab_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

default_model_vocab_path.write_text(
    json.dumps(vocab_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

with vocab_tsv_path.open('w', encoding='utf-8') as handle:
    handle.write('id\ttoken\n')
    for idx, token in enumerate(itos):
        handle.write(f'{idx}\t{token}\n')

payload = {
    'model_state_dict': model.state_dict(),
    'config': {
        'context_size': CONTEXT_SIZE,
        'embedding_dim': 96,
        'hidden_dim': 160,
    },
    'vocab_file': vocab_json_path.name,
}

torch.save(payload, model_path)
print(f'Saved model weights:         {model_path}')
print(f'Saved vocabulary JSON:       {vocab_json_path}')
print(f'Saved vocabulary TSV:        {vocab_tsv_path}')
print(f'Updated model vocabulary at: {default_model_vocab_path}')


Saved artifact: /content/model/artifacts/small_poetry_gru.pt


In [ ]:
# Optional: reload vocabulary from separate file (for a fresh notebook session)
loaded_vocab = json.loads(vocab_json_path.read_text(encoding='utf-8'))
itos = loaded_vocab['tokens']
stoi = {token: idx for idx, token in enumerate(itos)}
PAD_ID = stoi['<pad>']
UNK_ID = stoi['<unk>']

print(f'Vocabulary reloaded from {vocab_json_path}')
print(f'Vocabulary size: {len(itos):,}')


In [11]:
# Inference helpers with optional rhyme + tempo constraints
VOWELS = set('aeiouy')


def syllable_count(word: str) -> int:
    count = sum(1 for ch in word.lower() if ch in VOWELS)
    return max(1, count)


def rhyme_key(word: str, n: int = 2) -> str:
    w = re.sub(r"[^a-z']", '', word.lower())
    return w[-n:] if len(w) >= n else w


def encode_context(context: str, context_size: int = CONTEXT_SIZE) -> torch.Tensor:
    tokens = WORD_RE.findall(context.lower())
    ids = [stoi.get(tok, UNK_ID) for tok in tokens][-context_size:]
    if len(ids) < context_size:
        ids = [PAD_ID] * (context_size - len(ids)) + ids
    return torch.tensor([ids], dtype=torch.long, device=DEVICE)


def predict_next_word(
    context: str,
    top_k: int = 5,
    rhyme_with: str | None = None,
    target_syllables: int | None = None,
    rhyme_weight: float = 1.3,
    tempo_weight: float = 0.5,
):
    model.eval()
    with torch.no_grad():
        x = encode_context(context)
        logits = model(x)[0]
        log_probs = torch.log_softmax(logits, dim=-1)

    # Consider a wider pool, then re-rank with rhyme/tempo constraints.
    pool_k = min(len(itos), max(top_k * 20, 100))
    candidate_vals, candidate_ids = torch.topk(log_probs, k=pool_k)

    scored = []
    for lp, idx in zip(candidate_vals.tolist(), candidate_ids.tolist()):
        token = itos[idx]
        if token in SPECIAL_TOKENS:
            continue

        score = lp
        if rhyme_with and rhyme_key(token) == rhyme_key(rhyme_with):
            score += rhyme_weight
        if target_syllables is not None:
            score -= tempo_weight * abs(syllable_count(token) - target_syllables)

        scored.append((token, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


In [12]:
# Demo
context = 'the silver moon above the'
print('Context:', context)
print('Unconstrained:', predict_next_word(context, top_k=5))
print('Rhyme with "night":', predict_next_word(context, top_k=5, rhyme_with='night'))
print('Target 1 syllable:', predict_next_word(context, top_k=5, target_syllables=1))


Context: the silver moon above the
Unconstrained: [('rest', -1.7622698545455933), ('sun', -3.146286964416504), ('hills', -3.500969409942627), ('water', -3.8039751052856445), ('sky', -3.9951539039611816)]
Rhyme with "night": [('rest', -1.7622698545455933), ('light', -2.9895164489746096), ('sun', -3.146286964416504), ('hills', -3.500969409942627), ('water', -3.8039751052856445)]
Target 1 syllable: [('rest', -1.7622698545455933), ('sun', -3.146286964416504), ('hills', -3.500969409942627), ('sky', -3.9951539039611816), ('stars', -4.196460247039795)]


## Next step for integration

After you train and save `model/artifacts/small_poetry_gru.pt`, you can update `model/__init__.py`
so `model.predict(context)` loads this artifact and uses `predict_next_word(...)` instead of the dummy scorer.
